# KernelPack RAG: From Naive to Hybrid

A step-by-step retrieval experiment on `kernelpack-python`.

Each section introduces **one fix** that addresses a specific, observed failure mode. The failure modes accumulate into a requirements list that drives the VDB recommendation.

**Three fixed benchmark queries run across all sections:**
1. *"how to create an RBF-FD differentiation matrix?"*
2. *"how to define an ellipse boundary and generate nodes?"*
3. *"how to step diffusion forward in time?"*

**Scoring note:** Sections 1–3 use cosine similarity (higher = better) or L2 distance (lower = better). Section 4 uses RRF score (higher = better). Don't compare absolute scores across sections — use **rank position** instead.

| Section | What changed | Chunks | Retrieval |
|---|---|---|---|
| 1 | Full-repo baseline | 185 fixed-line | Dense (MiniLM) |
| 2 | AST-based chunking | 520 function-level | Dense (MiniLM) + ChromaDB |
| 3 | Code embedding model | 520 function-level | Dense (UniXcoder) |
| 4 | Short-function filter + BM25 | 252 function-level | Hybrid RRF (UniXcoder + BM25) |

## 0. Setup

In [1]:
%pip install -q sentence-transformers tree-sitter tree-sitter-python chromadb rank-bm25


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from pathlib import Path

import chromadb
import tree_sitter_python as tspython
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tree_sitter import Language, Parser

# ── paths ──────────────────────────────────────────────────────────────────────
# Assumes kernelpack-python is cloned in the same parent directory as this repo.
# If not, update this path to point at your local kernelpack-python/src/kernelpack.
REPO_PATH = Path("../../kernelpack-python/src/kernelpack")
if not REPO_PATH.exists():
    raise FileNotFoundError(
        f"kernelpack-python not found at {REPO_PATH.resolve()}. "
        "Clone it as a sibling directory: git clone https://github.com/ShankarLab/kernelpack-python"
    )

# ── retrieval config ───────────────────────────────────────────────────────────
TOP_K     = 3   # results returned per query
MIN_LINES = 5   # Section 4: functions shorter than this are dropped

# ── benchmark queries (same across all sections) ───────────────────────────────
BENCHMARK_QUERIES = [
    "how to create an RBF-FD differentiation matrix?",
    "how to define an ellipse boundary and generate nodes?",
    "how to step diffusion forward in time?",
]

In [3]:
# Load all .py files from the repo
def load_docs(repo_path: Path) -> list[dict]:
    docs = []
    for path in sorted(repo_path.rglob("*.py")):
        docs.append({"path": str(path), "text": path.read_text(encoding="utf-8")})
    print(f"Loaded {len(docs)} files from {repo_path}")
    return docs

docs = load_docs(REPO_PATH)

Loaded 27 files from ../../kernelpack-python/src/kernelpack


## 1. Naive Baseline

**Pipeline:** read all `.py` files → split at fixed line count → embed with `all-MiniLM-L6-v2` → cosine similarity retrieval

**Known problems going in:**
- Fixed-line chunks cut across class and function boundaries — the most relevant code gets split mid-method
- Dense-only search is sensitive to surface form: `"rbffd"` (bare token) and `"RBF-FD"` (spelled out) return different results even though they mean the same thing
- No persistence — re-embeds everything on every run

This covers the MVP-0 concept (single file) scaled to the full repo (MVP-1).

In [4]:
# Fixed-line chunking: split each file every CHUNK_SIZE lines, attach source path
CHUNK_SIZE = 40

chunks_v1 = []
for doc in docs:
    lines = doc["text"].splitlines()
    for i in range(0, len(lines), CHUNK_SIZE):
        chunks_v1.append({
            "path": doc["path"],
            "text": "\n".join(lines[i : i + CHUNK_SIZE]),
        })

print(f"Total chunks: {len(chunks_v1)}  ({CHUNK_SIZE} lines each)")

Total chunks: 185  (40 lines each)


In [5]:
# Embed all chunks with all-MiniLM-L6-v2 (384-dim, general-purpose NL model)
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
model_v1     = SentenceTransformer("all-MiniLM-L6-v2")
embeddings_v1 = model_v1.encode([c["text"] for c in chunks_v1])
print(f"Embeddings shape: {embeddings_v1.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings shape: (185, 384)


In [6]:
def retrieve_v1(query: str, k: int = TOP_K) -> None:
    q      = model_v1.encode([query])[0]
    scores = np.dot(embeddings_v1, q) / (
        np.linalg.norm(embeddings_v1, axis=1) * np.linalg.norm(q)
    )
    top = np.argsort(scores)[::-1][:k]
    print(f"\nQUERY: \"{query}\"")
    print("=" * 60)
    for rank, idx in enumerate(top, 1):
        print(f"\n  Rank {rank} | cosine {scores[idx]:.4f}")
        print(f"  {chunks_v1[idx]['path']}")
        print("  " + "-" * 56)
        print("  " + chunks_v1[idx]["text"][:300].replace("\n", "\n  "))

In [7]:
# ── Failure mode demo: surface-form sensitivity ────────────────────────────
# Same intent, different token form → different results.
# This is why dense-only search on code is insufficient.
print("=== Token surface-form comparison ===")
for q in ["how to create rbffd matrix?",
          "how to create an RBF-FD differentiation matrix?"]:
    retrieve_v1(q)

=== Token surface-form comparison ===

QUERY: "how to create rbffd matrix?"

  Rank 1 | cosine 0.4195
  ../../kernelpack-python/src/kernelpack/geometry/core.py
  --------------------------------------------------------
  from __future__ import annotations
  
  from dataclasses import dataclass, field
  from itertools import product
  
  import numpy as np
  from scipy import linalg
  from scipy.spatial import Delaunay, cKDTree
  
  from kernelpack._numba import dense_distance_matrix, phs_kernel_matrix
  
  
  def distance_matrix(x: np.ndar

  Rank 2 | cosine 0.4169
  ../../kernelpack-python/src/kernelpack/_numba.py
  --------------------------------------------------------
              for d in range(x.shape[1]):
                  xc[i, d] = (x[i, d] - xm[d]) * inv_width
          return xm, xc, width
  
  
      @njit(cache=True, fastmath=True)
      def _build_augmented_rbf_lhs_numba(kernel: np.ndarray, poly: np.ndarray) -> np.ndarray:
          n = kernel.shape[0]
          npoly =

In [8]:
# ── Benchmark queries ──────────────────────────────────────────────────────
for q in BENCHMARK_QUERIES:
    retrieve_v1(q)


QUERY: "how to create an RBF-FD differentiation matrix?"

  Rank 1 | cosine 0.4087
  ../../kernelpack-python/src/kernelpack/_numba.py
  --------------------------------------------------------
              for d in range(x.shape[1]):
                  xc[i, d] = (x[i, d] - xm[d]) * inv_width
          return xm, xc, width
  
  
      @njit(cache=True, fastmath=True)
      def _build_augmented_rbf_lhs_numba(kernel: np.ndarray, poly: np.ndarray) -> np.ndarray:
          n = kernel.shape[0]
          npoly =

  Rank 2 | cosine 0.4084
  ../../kernelpack-python/src/kernelpack/poly/core.py
  --------------------------------------------------------
  
      def evaluate(self, x: np.ndarray, d: np.ndarray | None = None, assume_normalized: bool = False) -> np.ndarray:
          # Evaluate the full multivariate basis and then apply the chain-rule
          # scaling needed when the caller asked for derivatives in physical
          # coordinates instead of

  Rank 3 | cosine 0.4078
  ../../ker

### Section 1 Results

```
QUERY: "how to create an RBF-FD differentiation matrix?"
  Rank 1 | cosine 0.4087  kernelpack/_numba.py
  Rank 2 | cosine 0.4084  kernelpack/poly/core.py
  Rank 3 | cosine 0.4078  kernelpack/rbffd/core.py

QUERY: "how to define an ellipse boundary and generate nodes?"
  Rank 1 | cosine 0.3953  kernelpack/geometry/core.py
  Rank 2 | cosine 0.3783  kernelpack/solvers/pu_sl_advection.py   ← wrong
  Rank 3 | cosine 0.3759  kernelpack/nodes/core.py

QUERY: "how to step diffusion forward in time?"
  Rank 1 | cosine 0.5587  kernelpack/solvers/pu_sl_fd_advection_diffusion.py
  Rank 2 | cosine 0.5078  kernelpack/solvers/pu_sl_pu_advection_diffusion.py
  Rank 3 | cosine 0.4940  kernelpack/solvers/heterogeneous_multispecies_diffusion.py
```

**Analysis:**
- Q1: scores nearly identical (0.41, 0.41, 0.41) across three unrelated files — the model can't distinguish them. Fixed-line chunking split the relevant stencil assembly logic across chunk boundaries, so no chunk actually answers the question.
- Q2: Rank 1 is geometry (correct direction). Rank 2 is an advection solver (irrelevant). Low scores throughout.
- Q3: works. "diffusion" appears at the class level in every solver file — coarse NL maps well to file-level semantics even with naive chunking.

**Failure modes confirmed:**
- Fixed-line chunks split method bodies — relevant code is never fully in one chunk
- Dense-only search is sensitive to token surface form (`rbffd` ≠ "RBF-FD matrix")

**What still works:**
- High-level queries over class-level concepts (Q3)

## 2. Fix 1 — Semantic Chunking (tree-sitter)

**What changed:** replace fixed-line chunks with AST-based function-level chunks.

**Why:** a function is the natural semantic unit in code. It has a name, a signature, a docstring, and a coherent body. Chunking at function boundaries means each chunk is self-contained and attributable to a specific location in the codebase.

**Result:** 185 fixed-line chunks → 520 function-level chunks. Each chunk carries metadata: `file_path`, `class_name`, `function_name`, `start_line`, `end_line`.

**New problem exposed:** trivial getter methods (1–2 lines) are indexed as equal-weight chunks alongside 500-line implementations. The embedding model has no content to work with on a 1-liner.

**Also added:** ChromaDB for persistence — no re-embedding on every run.

> **Scoring note:** ChromaDB returns L2 distances — lower = more similar. Opposite of cosine similarity above.

In [9]:
# tree-sitter parser setup
PY_LANGUAGE = Language(tspython.language())
parser_ts   = Parser(PY_LANGUAGE)

def get_class_name(node):
    """Return the enclosing class name if the node is a method, else None."""
    if node.parent.type == "block" and node.parent.parent.type == "class_definition":
        return node.parent.parent.child_by_field_name("name").text.decode("utf-8")
    return None

def extract_chunks_v2(source: str, tree, path: str) -> list[dict]:
    """Walk the AST; return one chunk per function definition. No size filter."""
    chunks = []
    def walk(node):
        if node.type == "function_definition":
            chunks.append({
                "path": path,
                "function_name": node.child_by_field_name("name").text.decode("utf-8"),
                "class_name":    get_class_name(node),
                "text":          source[node.start_byte:node.end_byte],
                "start_line":    node.start_point[0],
                "end_line":      node.end_point[0],
            })
        for child in node.children:
            walk(child)
    walk(tree.root_node)
    return chunks

In [10]:
# Parse all files and build ChromaDB collection (MiniLM embeddings)
all_chunks_v2 = []
for doc in docs:
    tree = parser_ts.parse(doc["text"].encode("utf-8"))
    all_chunks_v2.extend(extract_chunks_v2(doc["text"], tree, doc["path"]))

# Deduplicate: nested functions can share line ranges
documents_v2, metadatas_v2, ids_v2 = [], [], []
seen = set()
for c in all_chunks_v2:
    fp  = str(Path(c["path"]).relative_to(REPO_PATH))
    cid = f"{fp}:{c['start_line']}-{c['end_line']}"
    if cid in seen:
        continue
    seen.add(cid)
    documents_v2.append(c["text"])
    metadatas_v2.append({
        "file_path":     fp,
        "class_name":    c["class_name"] or "",
        "function_name": c["function_name"],
        "start_line":    c["start_line"],
        "end_line":      c["end_line"],
    })
    ids_v2.append(cid)

ef_v2  = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client = chromadb.Client()
col_v2 = client.get_or_create_collection("kp-v2-minilm", embedding_function=ef_v2)
col_v2.add(documents=documents_v2, metadatas=metadatas_v2, ids=ids_v2)
print(f"Collection kp-v2-minilm: {col_v2.count()} chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Collection kp-v2-minilm: 520 chunks


In [11]:
def retrieve_chroma(query: str, collection, k: int = TOP_K) -> None:
    """Dense retrieval via ChromaDB. Returns L2 distances (lower = better)."""
    res = collection.query(query_texts=[query], n_results=k)
    print(f"\nQUERY: \"{query}\"")
    print("=" * 60)
    for i in range(k):
        meta = res["metadatas"][0][i]
        dist = res["distances"][0][i]
        cls  = f"{meta['class_name']}." if meta["class_name"] else ""
        print(f"\n  Rank {i+1} | L2 {dist:.4f}")
        print(f"  {meta['file_path']} — {cls}{meta['function_name']} (lines {meta['start_line']}–{meta['end_line']})")
        print("  " + "-" * 56)
        print("  " + res["documents"][0][i][:300].replace("\n", "\n  "))

In [12]:
for q in BENCHMARK_QUERIES:
    retrieve_chroma(q, col_v2)


QUERY: "how to create an RBF-FD differentiation matrix?"

  Rank 1 | L2 0.5326
  geometry/core.py — RBFLevelSet.evaluate_gradient (lines 264–275)
  --------------------------------------------------------
  def evaluate_gradient(self, xe: np.ndarray) -> np.ndarray:
          # Differentiate the RBF interpolant analytically with respect to each
          # physical coordinate.
          xe = np.asarray(xe, dtype=float)
          r = distance_matrix(xe, self.centers)
          dphi = self.m_spline_degree * r ** ma

  Rank 2 | L2 0.5424
  rbffd/core.py — phs_rbf (lines 329–330)
  --------------------------------------------------------
  def phs_rbf(r: np.ndarray, degree: int) -> np.ndarray:
          return phs_kernel_matrix(r, degree)

  Rank 3 | L2 0.5462
  rbffd/core.py — RBFStencil.initialize_geometry (lines 181–197)
  --------------------------------------------------------
  def initialize_geometry(self, x: np.ndarray, sp: StencilProperties) -> None:
          # Normalize the sten

### Section 2 Results

```
QUERY: "how to create an RBF-FD differentiation matrix?"
  Rank 1 | L2 0.5326  geometry/core.py — RBFLevelSet.evaluate_gradient (lines 264–275)
  Rank 2 | L2 0.5424  rbffd/core.py — phs_rbf (lines 329–330)              ← 2-line wrapper
  Rank 3 | L2 0.5462  rbffd/core.py — RBFStencil.initialize_geometry (181–197)

QUERY: "how to define an ellipse boundary and generate nodes?"
  Rank 1 | L2 0.5710  nodes/core.py — DomainNodeGenerator.generate_interior_nodes_from_geometry
  Rank 2 | L2 0.5973  solvers/pu_sl_advection.py — boundary_hit_on_segment  ← wrong
  Rank 3 | L2 0.6139  geometry/core.py — build_planar_parametric_nodes_2d

QUERY: "how to step diffusion forward in time?"
  Rank 1 | L2 0.4136  solvers/pu_sl_pu_advection_diffusion.py — PUSLPUAdvectionDiffusionSolver.diffusion_solver  ← 2-line getter
  Rank 2 | L2 0.4318  solvers/pu_sl_fd_advection_diffusion.py — PUSLFDAdvectionDiffusionSolver.init
  Rank 3 | L2 0.4340  solvers/pu_sl_fd_advection_diffusion.py — PUSLFDAdvectionDiffusionSolver.diffusion_solver  ← 2-line getter
```

**Analysis:**
- Q1: hits `rbffd/core.py` (right file) but `assemble_op` never appears. `phs_rbf` at Rank 2 is a 2-line wrapper — the model has almost nothing to embed.
- Q2: Rank 1 is correct direction. Rank 2 is still an irrelevant advection file.
- Q3: all three results are trivial getters and an `init` method. Right files, completely wrong functions. The one-liner problem is now clearly visible.

**What chunking fixed:** chunks are semantically coherent; metadata makes results attributable.

**New failure mode:** trivial 1–2 line getter methods rank on par with real implementations — the embedding model has no content to work with on a getter.

## 3. Fix 2 — Code-Specific Embedding Model

**What changed:** swap `all-MiniLM-L6-v2` (trained on NL) for `microsoft/unixcoder-base` (RoBERTa-based, trained on code across multiple tasks: code search, generation, summarization).

**Why:** `all-MiniLM-L6-v2` was not trained on code. It handles NL queries adequately via code-adjacent training data, but it can't reliably map a natural language query to code-specific terminology. UniXcoder was purpose-trained for this.

**Same 520 chunks. Same pipeline. Model swap only.**  
`max_seq_length` is capped at 512 tokens to stay within UniXcoder's context limit.

> **Scoring note:** still L2 distance — lower = better.

In [13]:
# Rebuild collection with UniXcoder embeddings
model_v3 = SentenceTransformer("microsoft/unixcoder-base")
model_v3.max_seq_length = 512

ef_v3 = SentenceTransformerEmbeddingFunction(model_name="microsoft/unixcoder-base")
ef_v3._model = model_v3   # inject configured model so max_seq_length is respected

col_v3 = client.get_or_create_collection("kp-v3-unixcoder", embedding_function=ef_v3)
col_v3.add(documents=documents_v2, metadatas=metadatas_v2, ids=ids_v2)
print(f"Collection kp-v3-unixcoder: {col_v3.count()} chunks")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Collection kp-v3-unixcoder: 520 chunks


In [14]:
# retrieve_chroma works unchanged — pass col_v3
for q in BENCHMARK_QUERIES:
    retrieve_chroma(q, col_v3)


QUERY: "how to create an RBF-FD differentiation matrix?"

  Rank 1 | L2 0.6115
  solvers/pu_sl_pu_advection_diffusion.py — PUSLPUAdvectionDiffusionSolver.diffusion_solver (lines 101–102)
  --------------------------------------------------------
  def diffusion_solver(self) -> PUDiffusionSolver:
          return self.diffusion

  Rank 2 | L2 0.6201
  solvers/pu_sl_fd_advection_diffusion.py — PUSLFDAdvectionDiffusionSolver.diffusion_solver (lines 101–102)
  --------------------------------------------------------
  def diffusion_solver(self) -> DiffusionSolver:
          return self.diffusion

  Rank 3 | L2 0.6580
  solvers/_common.py — make_assembler (lines 46–53)
  --------------------------------------------------------
  def make_assembler(assembler_spec: str, stencil_spec: str | Callable[[], object]) -> FDDiffOp | FDODiffOp:
      factory = resolve_stencil_factory(stencil_spec)
      name = str(assembler_spec).lower()
      if name in {"fd", "fddiffop", "standard"}:
          retu

### Section 3 Results

```
QUERY: "how to create an RBF-FD differentiation matrix?"
  Rank 1 | L2 0.6115  solvers/pu_sl_pu_advection_diffusion.py — diffusion_solver (lines 101–102)  ← 2-line getter
  Rank 2 | L2 0.6201  solvers/pu_sl_fd_advection_diffusion.py — diffusion_solver (lines 101–102)  ← 2-line getter
  Rank 3 | L2 0.6580  solvers/_common.py — make_assembler (lines 46–53)

QUERY: "how to define an ellipse boundary and generate nodes?"
  Rank 1 | L2 0.5746  nodes/core.py — clip_points_by_geometry (lines 442–471)       ✓
  Rank 2 | L2 0.5932  nodes/core.py — DomainNodeGenerator.build_domain_descriptor_from_geometry  ✓
  Rank 3 | L2 0.6024  nodes/core.py — DomainNodeGenerator.generate_interior_nodes_from_geometry  ✓

QUERY: "how to step diffusion forward in time?"
  Rank 1 | L2 0.5373  solvers/pu_sl_fd_advection_diffusion.py — set_step_size (lines 38–41)  ← trivial setter
  Rank 2 | L2 0.5373  solvers/pu_sl_pu_advection_diffusion.py — set_step_size (lines 38–41)  ← trivial setter
  Rank 3 | L2 0.5507  solvers/pu_sl_multispecies.py — forward_sl_step (lines 82–84)
```

**Analysis:**
- Q1: still broken. Top 2 are 1-liner `diffusion_solver` properties. The model has nothing to embed. `assemble_op` doesn't appear.
- Q2: **best result of the experiment so far.** All 3 ranks hit `nodes/core.py` and return the correct functions. The model swap measurably helped here.
- Q3: right files, still wrong functions. `set_step_size` (trivial setter) beats actual stepping methods. Same one-liner problem.

**What UniXcoder fixed:** Q2 — all 3 results now correct. The model understands geometry/node generation terminology.

**What it didn't fix:** Q1 and Q3 still fail because short throwaway methods pollute the index. The root problem is chunk quality, not the model.

## 4. Fix 3 — Chunk Filter + Hybrid Retrieval

**Two changes:**

**4a. Chunk filter:** drop functions shorter than `MIN_LINES = 5` before indexing.  
520 → 252 chunks. Getter methods and 1-line wrappers are gone.

**4b. Hybrid retrieval:** add BM25 keyword search alongside dense search, merged via RRF.

> **BM25** (Best Match 25) is a keyword ranking function — the standard behind most search engines. > It scores documents by term frequency weighted by inverse document frequency. > Here it catches exact identifier matches that dense search misses.

> **RRF (Reciprocal Rank Fusion)** merges two ranked lists by summing `1 / (rank + 60)` from each list. > The `60` is a damping constant that limits the advantage of very high ranks. Higher RRF score = better.

> **Scoring note:** results are now ranked by RRF (higher = better). > L2 from the dense leg is shown alongside. "BM25-only" = not in the top-10 dense results — keyword search alone surfaced it.

In [15]:
# Updated extract_chunks: same tree-sitter walk, now drops functions < MIN_LINES
def extract_chunks_v4(source: str, tree, path: str, min_lines: int = MIN_LINES):
    """Walk the AST; return (kept, dropped) split by min_lines threshold."""
    kept, dropped = [], []
    def walk(node):
        if node.type == "function_definition":
            start = node.start_point[0]
            end   = node.end_point[0]
            entry = {
                "path":          path,
                "function_name": node.child_by_field_name("name").text.decode("utf-8"),
                "class_name":    get_class_name(node),
                "text":          source[node.start_byte:node.end_byte],
                "start_line":    start,
                "end_line":      end,
            }
            (kept if (end - start) >= min_lines else dropped).append(entry)
        for child in node.children:
            walk(child)
    walk(tree.root_node)
    return kept, dropped

all_chunks_v4, all_dropped_v4 = [], []
for doc in docs:
    tree = parser_ts.parse(doc["text"].encode("utf-8"))
    kept, dropped = extract_chunks_v4(doc["text"], tree, doc["path"])
    all_chunks_v4.extend(kept)
    all_dropped_v4.extend(dropped)

print(f"Chunks kept   : {len(all_chunks_v4)}")
print(f"Chunks dropped: {len(all_dropped_v4)}  (functions < {MIN_LINES} lines)")

Chunks kept   : 252
Chunks dropped: 268  (functions < 5 lines)


In [16]:
# Build ChromaDB collection with filtered chunks + UniXcoder embeddings
documents_v4, metadatas_v4, ids_v4 = [], [], []
seen = set()
for c in all_chunks_v4:
    fp  = str(Path(c["path"]).relative_to(REPO_PATH))
    cid = f"{fp}:{c['start_line']}-{c['end_line']}"
    if cid in seen:
        continue
    seen.add(cid)
    documents_v4.append(c["text"])
    metadatas_v4.append({
        "file_path":     fp,
        "class_name":    c["class_name"] or "",
        "function_name": c["function_name"],
        "start_line":    c["start_line"],
        "end_line":      c["end_line"],
    })
    ids_v4.append(cid)

col_v4 = client.get_or_create_collection("kp-v4-hybrid", embedding_function=ef_v3)
col_v4.add(documents=documents_v4, metadatas=metadatas_v4, ids=ids_v4)
print(f"Collection kp-v4-hybrid: {col_v4.count()} chunks")

Collection kp-v4-hybrid: 252 chunks


In [17]:
# BM25 index over the same filtered document set (whitespace tokenization)
tokenized_corpus = [doc.split() for doc in documents_v4]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index built.")

BM25 index built.


In [18]:
def retrieve_v4(query: str, k: int = TOP_K) -> None:
    """Hybrid retrieval: dense (ChromaDB/UniXcoder) + BM25, merged via RRF."""
    # dense leg
    dense_res = col_v4.query(query_texts=[query], n_results=10)
    dense_ids = dense_res["ids"][0]
    dense_l2  = {cid: d for cid, d in zip(dense_res["ids"][0], dense_res["distances"][0])}

    # BM25 leg
    bm25_scores = bm25.get_scores(query.split())
    top_bm25    = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:10]
    bm25_ids    = [ids_v4[i] for i in top_bm25]

    # RRF fusion
    rrf: dict[str, float] = {}
    for rank, cid in enumerate(dense_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)
    for rank, cid in enumerate(bm25_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)

    top_ids = sorted(rrf, key=rrf.__getitem__, reverse=True)[:k]
    fetched  = col_v4.get(ids=top_ids, include=["documents", "metadatas"])

    print(f"\nQUERY: \"{query}\"")
    print("=" * 60)
    for i, (cid, doc, meta) in enumerate(
        zip(fetched["ids"], fetched["documents"], fetched["metadatas"]), 1
    ):
        cls    = f"{meta['class_name']}." if meta["class_name"] else ""
        l2_str = f"{dense_l2[cid]:.4f}" if cid in dense_l2 else "n/a (BM25-only)"
        print(f"\n  Rank {i} | RRF {rrf[cid]:.4f} | L2 {l2_str}")
        print(f"  {meta['file_path']} — {cls}{meta['function_name']} (lines {meta['start_line']}–{meta['end_line']})")
        print("  " + "-" * 56)
        print("  " + doc[:300].replace("\n", "\n  "))

In [19]:
for q in BENCHMARK_QUERIES:
    retrieve_v4(q)


QUERY: "how to create an RBF-FD differentiation matrix?"

  Rank 1 | RRF 0.0164 | L2 0.6852
  rbffd/core.py — RBFStencil.bc_op (lines 243–261)
  --------------------------------------------------------
  def bc_op(self, sp: StencilProperties, op: OpProperties, neu_coeff: float, dir_coeff: float, r_rhs: np.ndarray, x_subset: np.ndarray, x: np.ndarray, x_at_origin_subset: np.ndarray, x_at_origin: np.ndarray, nr_subset: np.ndarray) -> np.ndarray:
          # Mixed boundary rows are assembled as a linear 

  Rank 2 | RRF 0.0167 | L2 0.6580
  solvers/_common.py — make_assembler (lines 46–53)
  --------------------------------------------------------
  def make_assembler(assembler_spec: str, stencil_spec: str | Callable[[], object]) -> FDDiffOp | FDODiffOp:
      factory = resolve_stencil_factory(stencil_spec)
      name = str(assembler_spec).lower()
      if name in {"fd", "fddiffop", "standard"}:
          return FDDiffOp(factory)
      if name in {"fdo",

  Rank 3 | RRF 0.0167 | L2 n/a (

### Section 4 Results

*RRF scores are small by design (~0.01–0.03). Use rank position, not score magnitude.*

```
QUERY: "how to create an RBF-FD differentiation matrix?"
  Rank 1 | RRF 0.0164 | L2 0.6852   rbffd/core.py — RBFStencil.bc_op
  Rank 2 | RRF 0.0167 | L2 0.6580   solvers/_common.py — make_assembler
  Rank 3 | RRF 0.0167 | L2 n/a (BM25-only)   solvers/_common.py — gmres_with_fallback  ← wrong

QUERY: "how to define an ellipse boundary and generate nodes?"
  Rank 1 | RRF 0.0167 | L2 0.5746   nodes/core.py — clip_points_by_geometry            ✓
  Rank 2 | RRF 0.0164 | L2 0.5932   nodes/core.py — build_domain_descriptor_from_geometry  ✓
  Rank 3 | RRF 0.0167 | L2 n/a (BM25-only)   solvers/pu_sl_advection.py — validate_boundary_condition_configuration  ← regression

QUERY: "how to step diffusion forward in time?"
  Rank 1 | RRF 0.0167 | L2 n/a (BM25-only)   solvers/diffusion.py — DiffusionSolver.set_initial_state  ← BM25 regression ("state")
  Rank 2 | RRF 0.0315 | L2 0.5873             solvers/diffusion.py — DiffusionSolver.bdf2_step          ✓
  Rank 3 | RRF 0.0167 | L2 0.5777             solvers/pu_sl_advection.py — backward_sl_step             ← wrong file
```

**What chunk filtering fixed:**
- Trivial getters gone — `set_step_size`, `diffusion_solver` properties no longer pollute results
- `bdf2_step` now visible at Rank 2 in Q3 (it had no chance before)

**What BM25 introduced:**
- Q1 slightly improved — `bc_op` and `make_assembler` are at least adjacent to the right code
- Q2 Rank 3 regression: `validate_boundary_condition_configuration` surfaced via "boundary" token match
- Q3 Rank 1 regression: `set_initial_state` ranked first because "state" scores high across all diffusion code

**Q1 is still broken.** `assemble_op` has not appeared in any section. This is addressed in Section 5.

## 5. What's Still Broken: The Query Mismatch Problem

After four iterations, **Q1 has never returned `assemble_op`** — the function that actually assembles the RBF-FD differentiation matrix.

This is not a retrieval failure. It's a vocabulary gap:

| What the query says | What the codebase says |
|---|---|
| "RBF-FD differentiation matrix" | `assemble_op`, `FDDiffOp`, `RBFStencil` |
| "ellipse boundary" | `EmbeddedSurface`, `build_closed_geometric_model_ps` |
| "step diffusion forward" | `bdf1_step`, `bdf2_step`, `bdf3_step` |

Neither dense search (no embedding overlap) nor BM25 (no token overlap) can bridge this gap. The user's vocabulary and the codebase's naming conventions are disjoint.

**Two directions to fix this:**

**1. LLM-augmented metadata at index time:**  
Run an LLM over each function to generate a natural-language description, usage examples, and synonyms. Embed the description alongside (or instead of) the raw code. Now "how to create an RBF-FD differentiation matrix" has something to match against: *"assembles a sparse differentiation matrix using radial basis function stencils..."*

**2. Query rewriting at retrieval time:**  
Before querying the index, have an LLM translate the user's query into codebase vocabulary: *"how to create an RBF-FD differentiation matrix?"* → *"assemble_op FDDiffOp RBFStencil"*  
Lightweight but brittle — requires the LLM to know the codebase vocabulary.

Both are future work. This experiment isolates retrieval as the variable and shows exactly where retrieval runs out of runway. That boundary is the argument for LLM-in-the-loop indexing.

## 6. Retrieval Requirements → VDB Recommendation

### Requirements extracted from this experiment

| Requirement | Evidence |
|---|---|
| **Hybrid search (dense + sparse)** | Dense alone misses exact identifiers (Q1 throughout). BM25 alone over-rewards common domain terms (Q3 regression). Need both, properly calibrated. |
| **Persistent index** | Re-embedding 500+ chunks on every run is unusable. At scale (Trilinos, PETSc), re-indexing is not an option. |
| **Function-level metadata filtering** | We need to filter by `file_path`, `class_name`, `function_name` to scope retrieval (e.g., "only search solver files"). |
| **Native RRF or configurable fusion** | The manual RRF implementation here works but is not persisted and cannot be tuned per-query. |
| **Python client** | The full pipeline is Python. |
| **Scalable to larger codebases** | KernelPack is ~250 filtered chunks. Trilinos/PETSc will be orders of magnitude larger — need an HNSW-backed index, not in-memory numpy. |

### ChromaDB vs. Qdrant

| Feature | ChromaDB (used here) | Qdrant |
|---|---|---|
| Native hybrid search | dense only | sparse + dense built-in |
| Persistent storage | yes (disk mode) | yes |
| Metadata filtering | yes | yes |
| Native RRF | manual implementation required | built-in |
| BM25 / sparse vectors | external, in-memory | `models.SparseVector` |
| Scale | small-to-medium | production-grade (HNSW) |
| Python client | yes | yes |

**ChromaDB was the right choice for this experiment** — zero infrastructure, easy to iterate, good enough to surface all four failure modes.

**Qdrant is the right choice for production** — the BM25 leg here is bolted on manually: it's not persisted, not calibrated against the dense leg, and not tunable. Qdrant's native sparse+dense support collapses three moving parts into one tool call. That's the right foundation for the team's CRAG pipeline.

### Recommended next step

Port this pipeline to Qdrant:
1. Same 252 filtered chunks, same UniXcoder dense embeddings, same metadata
2. Add sparse vectors (BM25 or SPLADE) as a second vector field in Qdrant
3. Query with Qdrant's built-in hybrid search + RRF
4. Re-run the same 3 benchmark queries and compare rank positions

That becomes the "Qdrant baseline" — the retrieval foundation the team's CRAG layer sits on top of.